# 福島県 空間線量率データ フォーマッター

福島県公式サイトの環境放射能水準調査データ（2011年3月〜2012年3月）を取得・整形する。

**処理の流れ:**
1. 月次CSVを福島県ウェブサイトからダウンロード（`raw/` に保存）
2. 各CSVを解析（多セクション・複雑フォーマット対応）
3. 監視ポスト名 → 緯度経度を国土地理院APIで取得
4. 対応幅CSV（`name, lat, lon, 2011_03, …, 2012_03`）を `../fukushima.csv` に出力

**生成後の使い方:**
```bash
# 単月のボロノイ図（福島県に絞り込み）
uv run python main.py fukushima 2011_04 --prefecture 福島県

# 時系列動画
uv run python scripts/generate_voronoi_video.py fukushima
```

In [ ]:
import csv
import json
import re
import time
import urllib.parse
from pathlib import Path

import pandas as pd
import requests

# ── パス ────────────────────────────────────────────────
RAW_DIR = Path('raw')
CACHE_JSON = Path('geocode_cache.json')
OUT_CSV = Path('../fukushima.csv')

RAW_DIR.mkdir(exist_ok=True)

# ── ダウンロード対象URL ──────────────────────────────────
BASE_URL = 'https://www.pref.fukushima.lg.jp/sec_file/monitoring/m-2/'

MONTHLY_FILES = [
    ('2011_03', 'zenken0311-0331.csv'),
    ('2011_04', 'zenken0401-0430.csv'),
    ('2011_05', 'zenken0501-0531.csv'),
    ('2011_06', 'zenken0601-0630.csv'),
    ('2011_07', 'zenken0701-0731.csv'),
    ('2011_08', 'zenken0801-0831.csv'),
    ('2011_09', 'zenken0901-0930.csv'),
    ('2011_10', 'zenken1001-1031.csv'),
    ('2011_11', 'zenken1101-1130.csv'),
    ('2011_12', 'zenken1201-1231.csv'),
    ('2012_01', 'zenken2012-0101-0131.csv'),
    ('2012_02', 'zenken2012-0201-0229.csv'),
    ('2012_03', 'zenken2012-0301-0331.csv'),
]

GSI_GEOCODE_URL = 'https://msearch.gsi.go.jp/address-search/AddressSearch?q='
API_WAIT_SEC = 0.4

print(f'対象: {len(MONTHLY_FILES)} ヶ月分')

対象: 13 ヶ月分


In [ ]:
# ── 1. CSVダウンロード ───────────────────────────────────
def download_csv(filename: str, dest: Path) -> bool:
    """福島県サイトからCSVをダウンロードして保存する。既存ファイルはスキップ。"""
    if dest.exists():
        print(f'  スキップ（既存）: {dest.name}')
        return True
    url = BASE_URL + filename
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
        print(f'  ✓ {dest.name} ({len(resp.content)//1024} KB)')
        return True
    except Exception as e:
        print(f'  ✗ {filename}: {e}')
        return False


print('CSVダウンロード開始...')
for period, filename in MONTHLY_FILES:
    download_csv(filename, RAW_DIR / f'{period}.csv')

downloaded = list(RAW_DIR.glob('*.csv'))
print(f'\n取得済み: {len(downloaded)} ファイル')

CSVダウンロード開始...
  ✓ 2011_03.csv (29 KB)
  ✓ 2011_04.csv (65 KB)
  ✓ 2011_05.csv (68 KB)
  ✓ 2011_06.csv (95 KB)
  ✓ 2011_07.csv (96 KB)
  ✓ 2011_08.csv (97 KB)
  ✓ 2011_09.csv (93 KB)
  ✓ 2011_10.csv (95 KB)
  ✓ 2011_11.csv (92 KB)
  ✓ 2011_12.csv (95 KB)
  ✓ 2012_01.csv (95 KB)
  ✓ 2012_02.csv (87 KB)
  ✓ 2012_03.csv (95 KB)

取得済み: 13 ファイル


In [ ]:
# ── 2. CSVパーサー ───────────────────────────────────────
# 福島県のCSVは複数の地域セクション（県北・県中・会津など）が
# 1ファイルに連結されており、各セクションに異なる監視ポストが並ぶ。
# セクション区切りは「方向及び距離」行で識別する。

def clean_cell(cell: str) -> str:
    """セル文字列から改行・余分なスペースを除去する。"""
    return cell.replace('\n', ' ').replace('\r', '').strip()


def extract_float(val_str: str) -> float | None:
    """'1.21' および '1.12 (9時30分)' の両形式から数値を抽出する。

    旧形式（〜2011年5月）は純粋な数字のみ。
    新形式（2011年6月〜）は "値 (時刻)" 形式。
    """
    m = re.match(r"^\s*([\d.]+)", str(val_str).strip())
    return float(m.group(1)) if m else None


def parse_fukushima_csv(filepath: Path) -> dict[str, list[float]]:
    """月次CSVを解析し {監視ポスト名: [計測値, ...]} を返す。

    旧形式（〜5月）: 値のみ "1.21"
    新形式（6月〜）: 値＋時刻 "1.12 (9時30分)" — extract_float で吸収。
    """
    rows = []
    with open(filepath, encoding='cp932', errors='replace') as f:
        for row in csv.reader(f):
            rows.append([clean_cell(cell) for cell in row])

    station_values: dict[str, list[float]] = {}

    # 「方向及び距離」行のインデックスを全て取得 → セクション区切り
    dist_row_indices = [
        i for i, row in enumerate(rows)
        if len(row) > 0 and '方向及び距離' in row[0]
    ]

    for sec_num, dist_idx in enumerate(dist_row_indices):
        # 監視ポスト名行はdist行の1つ前
        name_row_idx = dist_idx - 1
        if name_row_idx < 0:
            continue

        raw_names = rows[name_row_idx][2:]  # 先頭2列をスキップ
        station_names = [n if n else None for n in raw_names]

        # このセクションの終端を決定
        if sec_num + 1 < len(dist_row_indices):
            sec_end = dist_row_indices[sec_num + 1]
        else:
            sec_end = len(rows)

        # データ行（1回目 / 2回目）を収集
        for row in rows[dist_idx + 1: sec_end]:
            if len(row) < 3 or row[1] not in ('1回目', '2回目'):
                continue
            values = row[2: 2 + len(station_names)]
            for name, val_str in zip(station_names, values):
                if not name:
                    continue
                val = extract_float(val_str)
                if val is not None:
                    station_values.setdefault(name, []).append(val)

    return station_values


# 動作テスト（旧形式）
test_file = RAW_DIR / '2011_04.csv'
if test_file.exists():
    test = parse_fukushima_csv(test_file)
    print(f'2011_04（旧形式）: {len(test)} 監視ポスト検出')
    for name, vals in list(test.items())[:3]:
        print(f'  {name}: {len(vals)}件 / 平均 {sum(vals)/len(vals):.3f} μSv/h')

# 動作テスト（新形式）
test_file2 = RAW_DIR / '2011_06.csv'
if test_file2.exists():
    test2 = parse_fukushima_csv(test_file2)
    print(f'2011_06（新形式）: {len(test2)} 監視ポスト検出')

2011_04: 61 監視ポスト検出
  国見町 役場: 60件 / 平均 0.796 μSv/h
  福島北警察署桑折分庁舎 (桑折町): 60件 / 平均 1.229 μSv/h
  伊達 市役所: 60件 / 平均 1.330 μSv/h
  農業総合センター 果樹研究所 (福島市): 60件 / 平均 1.139 μSv/h
  福島 市役所: 60件 / 平均 1.649 μSv/h


In [ ]:
# ── 3. 全月データを解析 ──────────────────────────────────
# period -> {station_name: monthly_avg}
monthly_data: dict[str, dict[str, float]] = {}

for period, _ in MONTHLY_FILES:
    csv_path = RAW_DIR / f'{period}.csv'
    if not csv_path.exists():
        print(f'⚠ スキップ（ファイルなし）: {period}')
        continue
    station_vals = parse_fukushima_csv(csv_path)
    monthly_avg = {
        name: sum(vals) / len(vals)
        for name, vals in station_vals.items()
        if vals
    }
    monthly_data[period] = monthly_avg
    print(f'✓ {period}: {len(monthly_avg)} 監視ポスト')

# 全期間を通じたユニークな監視ポスト名
all_stations = sorted({name for d in monthly_data.values() for name in d})
print(f'\n全監視ポスト数: {len(all_stations)}')

✓ 2011_03: 61 監視ポスト
✓ 2011_04: 61 監視ポスト
✓ 2011_05: 61 監視ポスト
✓ 2011_06: 0 監視ポスト
✓ 2011_07: 0 監視ポスト
✓ 2011_08: 0 監視ポスト
✓ 2011_09: 0 監視ポスト
✓ 2011_10: 0 監視ポスト
✓ 2011_11: 0 監視ポスト
✓ 2011_12: 0 監視ポスト
✓ 2012_01: 0 監視ポスト
✓ 2012_02: 0 監視ポスト
✓ 2012_03: 0 監視ポスト

全監視ポスト数: 62


In [5]:
# ── 4. 国土地理院APIでジオコーディング ────────────────────
cache: dict[str, tuple[float, float] | None] = {}
if CACHE_JSON.exists():
    with CACHE_JSON.open(encoding='utf-8') as f:
        raw = json.load(f)
    cache = {k: tuple(v) if v else None for k, v in raw.items()}
    print(f'キャッシュ読み込み: {len(cache)} 件')


def extract_geocode_query(station_name: str) -> str:
    """監視ポスト名からジオコーディング用クエリを生成する。

    例:
      '福島北警察署桑折分庁舎 (桑折町)' → '福島県桑折町'
      '伊達市役所' → '福島県伊達市役所'
    """
    # 括弧内に市区町村名が含まれる場合はそちらを優先
    paren_match = re.search(r'[（(]([^）)]+[市町村区郡])[）)]', station_name)
    if paren_match:
        return '福島県' + paren_match.group(1)
    return '福島県' + station_name


def fetch_coords_gsi(query: str) -> tuple[float, float] | None:
    """国土地理院APIで住所→(lat, lon)を取得する。"""
    try:
        resp = requests.get(
            GSI_GEOCODE_URL + urllib.parse.quote(query), timeout=10
        )
        data = resp.json()
        if data:
            lon, lat = data[0]['geometry']['coordinates']
            return float(lat), float(lon)
    except Exception as e:
        print(f'    API エラー ({query}): {e}')
    return None


# 未キャッシュのポストだけ API 呼び出し
to_fetch = [s for s in all_stations if s not in cache]
print(f'新規ジオコーディング: {len(to_fetch)} 件')

for station in to_fetch:
    query = extract_geocode_query(station)
    print(f'  → {station} ({query}) ...', end=' ', flush=True)
    coords = fetch_coords_gsi(query)
    if coords is None and '福島県' + station != query:
        # フォールバック: 監視ポスト名全体で再検索
        coords = fetch_coords_gsi('福島県' + station)
    cache[station] = coords
    print(f'✓ {coords}' if coords else '✗ 取得失敗')
    time.sleep(API_WAIT_SEC)

with CACHE_JSON.open('w', encoding='utf-8') as f:
    json.dump(cache, f, ensure_ascii=False, indent=2)
print(f'\nキャッシュ保存: {len(cache)} 件')

failed = [s for s in all_stations if not cache.get(s)]
print(f'座標取得失敗: {len(failed)} 件 → {failed[:10]}')

新規ジオコーディング: 62 件
  → いわき市 三和支所 (福島県いわき市 三和支所) ... ✓ (37.20343, 140.679398)
  → いわき市 勿来支所 (福島県いわき市 勿来支所) ... ✓ (36.883339, 140.764511)
  → いわき市 四倉支所 (福島県いわき市 四倉支所) ... ✓ (37.113899, 140.981766)
  → いわき市 小名浜支所 (福島県いわき市 小名浜支所) ... ✓ (36.944984, 140.887344)
  → いわき市 小川支所 (福島県いわき市 小川支所) ... ✓ (37.123001, 140.868484)
  → いわき市 田人支所 (福島県いわき市 田人支所) ... ✓ (36.96851, 140.653137)
  → いわき市　　中央台南　　小学校 (福島県いわき市　　中央台南　　小学校) ... ✓ (37.010551, 140.925964)
  → ふくしま自治研修センター (福島市) (福島県福島市) ... ✓ (37.76083, 140.474747)
  → ビッグパレットふくしま (福島県ビッグパレットふくしま) ... ✓ (37.750031, 140.467773)
  → 三島町 役場 (福島県三島町 役場) ... ✓ (37.470276, 139.644455)
  → 三春町 役場 (福島県三春町 役場) ... ✓ (37.440498, 140.493393)
  → 下郷町 役場 (福島県下郷町 役場) ... ✓ (37.255554, 139.872238)
  → 中島村 役場 (福島県中島村 役場) ... ✓ (37.148819, 140.350204)
  → 二本松 市役所 (福島県二本松 市役所) ... ✓ (37.584858, 140.431213)
  → 二本松市 東和支所 (福島県二本松市 東和支所) ... ✓ (37.533527, 140.595886)
  → 伊達 市役所 (福島県伊達 市役所) ... ✓ (37.819103, 140.562958)
  → 会津坂下町役場 (福島県会津坂下町役場) ... ✓ (37.561474, 139.821701)


In [6]:
# ── 5. 対応幅CSV出力 ────────────────────────────────────
# 座標が取得できたポストだけ使用
valid_stations = [
    s for s in all_stations if cache.get(s) is not None
]

records = []
for station in valid_stations:
    lat, lon = cache[station]
    row = {'name': station, 'lat': lat, 'lon': lon}
    for period, _ in MONTHLY_FILES:
        row[period] = monthly_data.get(period, {}).get(station, None)
    records.append(row)

df_out = pd.DataFrame(records)
period_cols = [p for p, _ in MONTHLY_FILES]

# どの期間にも値がない行を除外
df_out = df_out.dropna(subset=period_cols, how='all')

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(OUT_CSV, index=False, encoding='utf-8')

print(f'出力完了: {OUT_CSV.resolve()}')
print(f'監視ポスト数: {len(df_out)}')
print(f'期間列: {period_cols}')
print()
print('--- 2011年4月の上位10地点 ---')
print(
    df_out[['name', '2011_04']]
    .dropna()
    .sort_values('2011_04', ascending=False)
    .head(10)
    .to_string(index=False)
)

出力完了: /Users/rosalina_0106/Develop/ginder/data/fukushima/fukushima.csv
監視ポスト数: 62
期間列: ['2011_03', '2011_04', '2011_05', '2011_06', '2011_07', '2011_08', '2011_09', '2011_10', '2011_11', '2011_12', '2012_01', '2012_02', '2012_03']

--- 2011年4月の上位10地点 ---
                name  2011_04
         川俣町 山木屋 郵便局 3.083750
             二本松 市役所 1.990167
              福島 市役所 1.648833
         福島県農業総合センター 1.607000
               郡山市役所 1.544333
              本宮 市役所 1.400667
              伊達 市役所 1.329833
              天栄村 役場 1.278667
   福島北警察署桑折分庁舎 (桑折町) 1.229333
農業総合センター 果樹研究所 (福島市) 1.138500
